# Lesson 6a: Function Approximation - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand the Need for Function Approximation** - Scaling beyond tabular methods
2. **Master Value Function Approximation** - Approximating V(s) and Q(s,a)
3. **Learn Gradient-Based Methods** - Stochastic gradient descent for RL
4. **Understand Linear Function Approximation** - Features and weights
5. **Learn Semi-Gradient Methods** - TD with function approximation
6. **Understand the Deadly Triad** - Convergence challenges
7. **Master Feature Construction** - Tile coding, RBFs, polynomials

Function approximation enables RL to scale from simple gridworlds to complex continuous domains.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. The Need for Function Approximation

### Limitations of Tabular Methods

**Tabular RL** (Lessons 1-5):
- Store separate value for each state (or state-action pair)
- Works perfectly for small, discrete state spaces
- Update one state without affecting others

**Problems**:

**1. Memory Explosion**
- States grow exponentially with dimensions
- Example: 10 variables × 10 values each = 10^10 states
- Impossible to store in memory

**2. Computational Explosion**
- Need to visit each state many times
- Large spaces: never visit most states
- Learning extremely slow

**3. Continuous States**
- Infinite states (e.g., robot position x ∈ ℝ)
- Cannot enumerate all states
- Tabular methods fundamentally cannot work

**4. No Generalization**
- Similar states treated independently
- Learn V(s=5) tells us nothing about V(s=5.001)
- Wasteful and slow

### Function Approximation Solution

**Key idea**: Approximate value function with parameterized function

$$\hat{v}(s, \mathbf{w}) \approx v_\pi(s)$$

$$\hat{q}(s, a, \mathbf{w}) \approx q_\pi(s, a)$$

where $\mathbf{w} \in \mathbb{R}^d$ is weight vector (d << |S|)

**Benefits**:
- ✅ **Compact representation**: Store d weights instead of |S| values
- ✅ **Generalization**: Similar states get similar values
- ✅ **Continuous states**: Can handle any state space
- ✅ **Learning from samples**: Update weights, affect many states

### Examples

**Linear function**:
$$\hat{v}(s, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s) = \sum_{i=1}^d w_i x_i(s)$$

where $\mathbf{x}(s)$ is feature vector

**Neural network**:
$$\hat{v}(s, \mathbf{w}) = f_{NN}(s; \mathbf{w})$$

where $f_{NN}$ is neural network with parameters $\mathbf{w}$

### Trade-offs

**Tabular**:
- ✅ Guaranteed convergence (under assumptions)
- ✅ No approximation error
- ❌ Cannot scale
- ❌ No generalization

**Function Approximation**:
- ✅ Scales to large/continuous spaces
- ✅ Automatic generalization
- ❌ Approximation error
- ❌ Convergence not guaranteed (in general)
- ❌ Requires feature engineering (except deep RL)

## 2. Value Function Approximation Framework

### General Setting

**Goal**: Find weights $\mathbf{w}$ such that:

$$\hat{v}(s, \mathbf{w}) \approx v_\pi(s) \quad \forall s \in \mathcal{S}$$

**Objective**: Minimize error over state distribution

$$\overline{VE}(\mathbf{w}) = \sum_{s \in \mathcal{S}} \mu(s) [v_\pi(s) - \hat{v}(s, \mathbf{w})]^2$$

where $\mu(s)$ is state distribution (how much we care about each state)

**Common choices for μ(s)**:
- On-policy distribution: $\mu(s) = $ fraction of time spent in s under π
- Uniform: $\mu(s) = 1/|\mathcal{S}|$ (equal weight to all states)
- Start-state distribution: Weight states by how often episodes start there

### Stochastic Gradient Descent

**Idea**: Update weights to reduce error

**Gradient of squared error**:
$$\nabla [v_\pi(s) - \hat{v}(s, \mathbf{w})]^2 = -2[v_\pi(s) - \hat{v}(s, \mathbf{w})] \nabla \hat{v}(s, \mathbf{w})$$

**SGD update**:
$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha [v_\pi(s) - \hat{v}(s, \mathbf{w}_t)] \nabla \hat{v}(s, \mathbf{w}_t)$$

Equivalently:
$$\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{\alpha}{2} \nabla [v_\pi(s) - \hat{v}(s, \mathbf{w}_t)]^2$$

**Key insight**: We don't know $v_\pi(s)$! That's what we're trying to learn.

**Solution**: Replace $v_\pi(s)$ with target:
- Monte Carlo: $G_t$ (return)
- TD(0): $R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w})$ (bootstrap)
- TD(λ): $G_t^\lambda$ (λ-return)

### General Gradient-Descent Update

$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha [U_t - \hat{v}(S_t, \mathbf{w}_t)] \nabla \hat{v}(S_t, \mathbf{w}_t)$$

where $U_t$ is **target** (estimate of $v_\pi(S_t)$)

**MC target**: $U_t = G_t$
- Unbiased
- High variance

**TD target**: $U_t = R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w})$
- Biased (uses current estimate)
- Low variance
- **Semi-gradient** (doesn't differentiate through target)

### Convergence Guarantees

**Theorem 6.1** (MC Convergence):

If $U_t = G_t$ (MC target) and:
1. Step sizes satisfy Robbins-Monro: $\sum_t \alpha_t = \infty$, $\sum_t \alpha_t^2 < \infty$

Then: $\mathbf{w}_t \to \mathbf{w}^*$ where $\mathbf{w}^*$ minimizes $\overline{VE}$

**For linear functions**: Converges to global optimum

**For non-linear**: Converges to local minimum

**Theorem 6.2** (TD Convergence - Linear Case):

For linear $\hat{v}(s, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s)$ with TD target:

Converges to $\mathbf{w}_{TD}$ such that:

$$\overline{VE}(\mathbf{w}_{TD}) \leq \frac{1}{1 - \gamma} \min_\mathbf{w} \overline{VE}(\mathbf{w})$$

**Interpretation**: TD solution at most $\frac{1}{1-\gamma}$ worse than best linear approximation

**For γ = 0.9**: TD within 10× of best

**For γ = 0.99**: TD within 100× of best

**Note**: MC finds best approximation, TD finds good (not optimal) approximation

## 3. Linear Function Approximation

### Definition

$$\hat{v}(s, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s) = \sum_{i=1}^d w_i x_i(s)$$

where:
- $\mathbf{x}(s) = [x_1(s), x_2(s), ..., x_d(s)]^T$ is **feature vector**
- $\mathbf{w} = [w_1, w_2, ..., w_d]^T$ is **weight vector**
- d is number of features (d << |S|)

**Key property**: Linear in weights (not necessarily in state!)

### Gradient

$$\nabla \hat{v}(s, \mathbf{w}) = \nabla [\mathbf{w}^T \mathbf{x}(s)] = \mathbf{x}(s)$$

**Beautiful simplicity**: Gradient is just the feature vector!

### SGD Update (Linear Case)

$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha [U_t - \hat{v}(S_t, \mathbf{w}_t)] \mathbf{x}(S_t)$$

or equivalently:

$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha [U_t - \mathbf{w}_t^T \mathbf{x}(S_t)] \mathbf{x}(S_t)$$

### Action-Value Approximation

$$\hat{q}(s, a, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s, a)$$

**Update**:
$$\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha [U_t - \hat{q}(S_t, A_t, \mathbf{w}_t)] \mathbf{x}(S_t, A_t)$$

### Feature Engineering

**Critical choice**: How to construct features $\mathbf{x}(s)$?

**Example 1: State Aggregation**

Group similar states:
$$x_i(s) = \begin{cases} 1 & \text{if } s \in \text{group } i \\ 0 & \text{otherwise} \end{cases}$$

All states in same group share weight → same value

**Example 2: Polynomial Features**

For state s = (s₁, s₂):
$$\mathbf{x}(s) = [1, s_1, s_2, s_1^2, s_1 s_2, s_2^2, ...]^T$$

**Example 3: Radial Basis Functions (RBFs)**

$$x_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma_i^2}\right)$$

where $c_i$ is center of RBF i, $\sigma_i$ is width

**Example 4: Tile Coding** (very popular in RL)

- Overlay multiple offset grids (tilings)
- Each tiling partitions state space
- Feature active = 1 if state in that tile
- Provides coarse coding with generalization

### Why Linear?

**Advantages**:
✅ Simple to implement
✅ Efficient computation (O(d) per update)
✅ Convergence guarantees (with TD)
✅ Well-understood theory
✅ Easy to interpret

**Disadvantages**:
❌ Limited expressiveness
❌ Requires manual feature engineering
❌ May not capture complex patterns

**Note**: Deep RL (Lessons 7-10) uses neural networks to automatically learn features

## 4. Semi-Gradient Methods

### The Bootstrap Dilemma

**TD target**:
$$U_t = R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w})$$

**Problem**: Target depends on $\mathbf{w}$!

**True gradient** would be:
$$\nabla [R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}) - \hat{v}(S_t, \mathbf{w})]^2$$

$$= 2[R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}) - \hat{v}(S_t, \mathbf{w})]$$
$$\quad \times [\gamma \nabla \hat{v}(S_{t+1}, \mathbf{w}) - \nabla \hat{v}(S_t, \mathbf{w})]$$

**Semi-gradient**: Ignore gradient through target

$$\nabla_{semi} = 2[R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}) - \hat{v}(S_t, \mathbf{w})] [-\nabla \hat{v}(S_t, \mathbf{w})]$$

**Justification**: Treat target as constant (like supervised learning)

### Semi-Gradient TD(0)

```
Input: policy π, α
Initialize w arbitrarily

For each episode:
  Initialize S
  
  For each step:
    A ~ π(·|S)
    Take A, observe R, S'
    
    w ← w + α[R + γ v̂(S', w) - v̂(S, w)] ∇v̂(S, w)
    
    S ← S'
    If S' terminal: break
```

**Linear case**:
```
w ← w + α[R + γ w^T x(S') - w^T x(S)] x(S)
```

### Semi-Gradient SARSA

```
Input: α, ε
Initialize w arbitrarily

For each episode:
  Initialize S
  A ~ ε-greedy(q̂(S, ·, w))
  
  For each step:
    Take A, observe R, S'
    A' ~ ε-greedy(q̂(S', ·, w))
    
    w ← w + α[R + γ q̂(S', A', w) - q̂(S, A, w)] ∇q̂(S, A, w)
    
    S ← S'; A ← A'
    If S' terminal: break
```

**Linear case**:
```
w ← w + α[R + γ w^T x(S', A') - w^T x(S, A)] x(S, A)
```

### Semi-Gradient Q-Learning

```
w ← w + α[R + γ max_a q̂(S', a, w) - q̂(S, A, w)] ∇q̂(S, A, w)
```

**Warning**: Off-policy + function approximation + bootstrapping = potential divergence!

### Semi-Gradient n-Step Methods

**n-step return**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n \hat{v}(S_{t+n}, \mathbf{w})$$

**Update**:
$$\mathbf{w}_{t+n} = \mathbf{w}_{t+n-1} + \alpha [G_t^{(n)} - \hat{v}(S_t, \mathbf{w}_{t+n-1})] \nabla \hat{v}(S_t, \mathbf{w}_{t+n-1})$$

### Why "Semi-Gradient"?

**True gradient descent**:
- Minimizes objective function
- Guaranteed to find local minimum
- Converges to stationary point

**Semi-gradient descent**:
- Not true gradient of any objective
- May not converge (off-policy case)
- Can diverge with function approximation
- **But**: Often works well in practice!
- **And**: Converges for linear on-policy (Theorem 6.2)

## 5. The Deadly Triad

### Three Ingredients of Instability

**Deadly Triad** (Sutton & Barto, 2018):

1. **Function Approximation**: Using $\hat{v}(s, \mathbf{w})$ instead of table
2. **Bootstrapping**: Using estimates as targets (TD methods)
3. **Off-Policy Learning**: Learning about different policy than following

**Any two** → Safe (convergent)

**All three** → Danger! (can diverge)

### Examples of Safe Combinations

**MC + Function Approximation + Off-Policy**:
- ✅ MC uses true returns (no bootstrapping)
- ✅ Converges (with importance sampling)

**TD + Function Approximation + On-Policy**:
- ✅ SARSA, Actor-Critic converge (linear case)
- ✅ Semi-gradient TD converges (Theorem 6.2)

**TD + Bootstrapping + Tabular**:
- ✅ Q-Learning converges (tabular)
- ✅ All previous tabular methods safe

### The Dangerous Combination

**Q-Learning + Function Approximation**:
- Off-policy: Learn greedy, act ε-greedy
- Bootstrapping: Use max Q for target
- Function approximation: Generalize across states

**Result**: Can diverge! (Even with linear approximation)

### Classic Counter-Example: Baird's Example

**Setup**:
- 7 states (6 non-terminal + 1 terminal)
- Linear function approximation
- Off-policy learning

**Behavior policy**: Random (dashed lines)

**Target policy**: Solid lines

**Result**: Weights diverge to infinity!

```
Feature vectors chosen such that:
- Semi-gradient Q-learning diverges
- Weights grow without bound
- Even though problem is simple!
```

### Why Does It Diverge?

**Intuition**:
1. Update increases V(s) for some state s
2. Function approximation generalizes to other states
3. Targets for other states increase
4. Updates increase values further
5. **Positive feedback loop** → divergence

**Off-policy** breaks the balance:
- Visiting states according to behavior policy
- Updating according to target policy
- Updates don't match state distribution
- Generalization causes instability

### Solutions to the Deadly Triad

**1. Avoid it**:
- Use on-policy methods (SARSA, A3C)
- Use MC instead of TD
- Use tabular methods

**2. Experience Replay** (DQN):
- Store transitions in buffer
- Sample uniformly for updates
- Breaks correlations
- Improves stability (Lesson 7)

**3. Target Networks** (DQN):
- Use separate network for targets
- Update slowly
- Reduces moving target problem

**4. Gradient TD Methods**:
- GTD, GTD2, TDC
- True gradient descent
- Convergence guarantees
- More complex, slower

**5. Emphatic TD**:
- Weight updates by state visitation
- Corrects distribution mismatch
- Convergent

### Practical Guidance

**Be Careful When Using**:
- Q-learning with function approximation
- Any off-policy + TD + FA combination

**Monitor**:
- Weight magnitudes (check for explosion)
- Value estimates (check for instability)
- Policy performance

**Default Choice**:
- On-policy methods (SARSA, A3C) for safety
- DQN with experience replay for off-policy
- PPO for both safety and performance (Lesson 9)

## 6. Feature Construction Methods

### Tile Coding

**Idea**: Multiple overlapping grid tilings

**Construction**:
1. Create k tilings (grids)
2. Offset each tiling by different amount
3. For state s, find which tile in each tiling
4. Set those k features to 1, rest to 0

**Example** (1D state space, 3 tilings):

```
Tiling 1: [--|--|--|--|--]
Tiling 2:  [--|--|--|--|--]
Tiling 3:   [--|--|--|--|--]

State s = 2.3:
- Tiling 1: tile 2 active
- Tiling 2: tile 2 active  
- Tiling 3: tile 1 active

Feature vector: x(s) = [0,0,1,0,0, 0,0,1,0,0, 0,1,0,0,0]
```

**Properties**:
- ✅ Exact representation within tiles
- ✅ Smooth generalization across tile boundaries
- ✅ Adjustable resolution (number of tiles)
- ✅ Adjustable generalization (number of tilings)
- ✅ Efficient: Only k active features
- ✅ Easy to implement

**Generalization**:
- States in same tile: share value (generalize)
- Nearby states: share some tiles → partial generalization
- Distant states: share no tiles → no generalization

**Update**: Only k weights change (those with active features)

$$\mathbf{w} \leftarrow \mathbf{w} + \frac{\alpha}{k} [U - \hat{v}(s, \mathbf{w})] \mathbf{x}(s)$$

(Divide by k to normalize learning rate)

### Radial Basis Functions (RBFs)

**Gaussian RBF**:
$$x_i(s) = \exp\left(-\frac{\|s - c_i\|^2}{2\sigma_i^2}\right)$$

**Properties**:
- Feature i = 1 when s = $c_i$ (center)
- Decays to 0 as s moves away from $c_i$
- Width $\sigma_i$ controls generalization

**Multiple RBFs**: Place centers throughout state space

$$\hat{v}(s, \mathbf{w}) = \sum_{i=1}^k w_i \exp\left(-\frac{\|s - c_i\|^2}{2\sigma_i^2}\right)$$

**Advantages**:
- ✅ Smooth approximation
- ✅ Differentiable (useful for policy gradient)
- ✅ Local generalization

**Disadvantages**:
- ❌ All features active (computational cost)
- ❌ Curse of dimensionality (need many centers)

### Polynomial Features

**For state** $s = (s_1, s_2, ..., s_n)$:

**Order 1** (linear):
$$\mathbf{x}(s) = [1, s_1, s_2, ..., s_n]^T$$

**Order 2** (quadratic):
$$\mathbf{x}(s) = [1, s_1, s_2, ..., s_n, s_1^2, s_1 s_2, ..., s_n^2]^T$$

**Order p**:
All monomials up to degree p

**Number of features**:
$$d = \binom{n + p}{p}$$

**Example**: n=5 dimensions, p=3 → d = 56 features

**Properties**:
- ✅ Can approximate any smooth function (high enough p)
- ✅ Global approximation
- ❌ Curse of dimensionality (d grows fast)
- ❌ Can overfit with high p

### Fourier Basis

**Idea**: Use cosine functions as features

$$x_i(s) = \cos(\pi \mathbf{c}_i^T s)$$

where $\mathbf{c}_i$ is coefficient vector

**Properties**:
- ✅ Can approximate periodic functions
- ✅ Well-studied in signal processing
- ✅ Adaptive learning rates per feature

### Comparison

| Method | Local | Smooth | Efficient | Dimensionality |
|--------|-------|--------|-----------|----------------|
| Tile Coding | ✓ | ~ | ✓✓ | Good |
| RBF | ✓ | ✓✓ | ~ | Poor |
| Polynomial | ✗ | ✓✓ | ✓ | Poor |
| Fourier | ✗ | ✓✓ | ✓ | Moderate |

**Recommendation**:
- **Low dimensions** (1-3): Any method works
- **Medium dimensions** (4-10): Tile coding
- **High dimensions** (>10): Deep learning (Lessons 7-10)

### Automated Feature Construction

**Deep Learning** (upcoming lessons):
- Neural networks learn features automatically
- No manual engineering needed
- Scales to high dimensions
- Handles images, text, etc.

This is why deep RL is revolutionary!

## Summary

### Key Concepts

✅ **Function Approximation** - Scale RL beyond tabular methods  
✅ **Value Function Approximation** - $\hat{v}(s, \mathbf{w}) \approx v_\pi(s)$  
✅ **Gradient Descent** - Optimize weights to minimize error  
✅ **Linear Methods** - $\hat{v}(s, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s)$  
✅ **Semi-Gradient TD** - Bootstrap with function approximation  
✅ **Deadly Triad** - FA + Bootstrapping + Off-Policy → Danger!  
✅ **Feature Construction** - Tile coding, RBFs, polynomials  

### Critical Equations

**Linear value function**:
$$\hat{v}(s, \mathbf{w}) = \mathbf{w}^T \mathbf{x}(s)$$

**Gradient**:
$$\nabla \hat{v}(s, \mathbf{w}) = \mathbf{x}(s)$$

**SGD update**:
$$\mathbf{w} \leftarrow \mathbf{w} + \alpha [U_t - \hat{v}(S_t, \mathbf{w})] \mathbf{x}(S_t)$$

**Semi-gradient TD(0)**:
$$\mathbf{w} \leftarrow \mathbf{w} + \alpha [R + \gamma \hat{v}(S', \mathbf{w}) - \hat{v}(S, \mathbf{w})] \mathbf{x}(S)$$

### Convergence Summary

| Method | On-Policy | Off-Policy |
|--------|-----------|------------|
| MC + Linear | ✓ Optimal | ✓ Optimal (IS) |
| TD + Linear | ✓ Near-optimal | ✗ Can diverge |
| Q-Learning + Linear | ✓ Converges | ✗ Can diverge |

### Practical Recommendations

**For learning**:
1. Start with on-policy (SARSA)
2. Use linear function approximation
3. Try tile coding for features
4. Monitor for divergence

**For performance**:
1. Tune number of features
2. Adjust generalization (tile widths, σ)
3. Use appropriate learning rate
4. Consider n-step methods

**For production**:
1. Deep RL (upcoming) for complex domains
2. Linear methods for interpretability
3. Careful feature engineering

### What's Next?

**Lesson 6b** will implement:
- Tile coding from scratch
- RBF features
- Semi-gradient SARSA
- Mountain Car with linear FA
- Convergence comparison experiments

**Lessons 7-10** will cover **Deep RL**:
- DQN (Lesson 7)
- Policy Gradients (Lessons 8-9)
- Continuous Control (Lesson 10)
- Automated feature learning!

---

**Lesson 6a Complete!** 🎉

You now understand the theoretical foundations of function approximation, the bridge from tabular to deep reinforcement learning.